In [1]:
import polars as pl
from pathlib import Path

In [ ]:
#lf = pl.scan_parquet("/home/daam/llm_cite_write/output/run_15398/filtered.parquet/8.parquet", n_rows=10000)
#df = lf.collect()
#df.sort('corpusid', 'start').to_pandas()

In [11]:
# --- Configuration ---
CLASSIFIED_DIR = "/home/daam/llm_cite_write/output/run_15398/classified.parquet"

# Where to save the samples
SAMPLE_OUTPUT_DIR = Path("/home/daam/llm_cite_write/output/run_15398/samples_2/")
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# An estimate of the total number of unique papers in your dataset
TOTAL_PAPERS = 14_000_000 # Or your best estimate
HASH_SEED = 33

# --- Create the LazyFrame from your results ---
print(f"Scanning processed files from: {CLASSIFIED_DIR}")
lf = pl.scan_parquet(CLASSIFIED_DIR)

# --- Define the samples using the hashing method ---

# For ~1,000 papers (1 in 10,000)
auto_eval_lf = lf.filter(
    pl.col("corpusid").hash(seed=HASH_SEED) % 10000 == 0
)

# For ~100 papers (1 in 100,000)
manual_eval_lf = lf.filter(
    pl.col("corpusid").hash(seed=HASH_SEED) % 100000 == 0
)

# --- Execute and save the samples ---

print(f"Saving automatic evaluation sample...")
auto_eval_lf.sink_parquet(SAMPLE_OUTPUT_DIR / "auto_eval_sample.parquet")

print(f"Saving manual evaluation sample...")
manual_eval_lf.sink_parquet(SAMPLE_OUTPUT_DIR / "manual_eval_sample.parquet")

print("✅ Sampling complete.")

Scanning processed files from: /home/daam/llm_cite_write/output/run_15398/classified.parquet
Saving automatic evaluation sample...
Saving manual evaluation sample...
✅ Sampling complete.


In [ ]:
# Filter out empty text and duplicated rows
# Filter out figure_table and ending rows

In [11]:
df = pl.read_parquet("/home/daam/llm_cite_write/output/run_15398/samples_2/manual_eval_sample.parquet")
df.write_csv("/home/daam/llm_cite_write/output/run_15398/samples_2/manual_eval_sample.csv")

In [12]:
df.filter(~pl.col('sec_label_extended').is_in(['figure_table', 'ending'])).to_pandas()

,corpusid,start,end_header,end_section,extracted,section_text,sec_label_extended
0,10607276,2579,2591,5187,introduction,Introduction\n\nIntensive care units (ICUs) pr...,introduction
1,10607276,5187,5197,5244,objectives,Objectives\n\nThe following were the aims of t...,introduction
2,10607276,5244,5295,5526,to quantify and benchmark levels of mobilisati...,To quantify and benchmark levels of mobilisati...,development
3,10607276,5526,5533,6048,methods,Methods\n\nDesign A prospective observational ...,methods
4,10607276,6048,6055,6058,setting,Setting\n\n\n,methods
...,...,...,...,...,...,...,...
1463,263372833,16925,16929,17132,1177,1177\n\nCell envelope homeostasis during stres...,development
1464,263372833,17132,17136,17132,1292,,development
1465,263372833,17132,17136,17195,1292,1292\n\nResidual plot of the rate data from Fi...,development
1466,263372833,17195,17199,17195,1303,,development


In [13]:
final_df = df.filter(
    (pl.col('section_text').str.strip_chars() != "") &
    (~pl.col('sec_label_extended').is_in(['figure_table', 'ending', 'other']))
)

In [15]:
final_df.write_csv("/home/daam/llm_cite_write/manual_annotations/sample_100.csv")